"""

This script demonstrates the overall data pipeline from FF-TEM Scan input to the output of preliminary analytics for Automated Rosette Analytics.

"""

# Install Ultralytics + Import Libraries

In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 99.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [10]:
import torch
import cv2
import numpy as np
import subprocess
import os
from ultralytics import YOLO
import csv
import math
from collections import defaultdict
import pandas as pd
import shutil
from IPython.display import display, clear_output
import ipywidgets as widgets
from pathlib import Path

# Data Input

In [3]:
def preprocess_image(input_path):
    """
    Reads a grayscale image from the specified path, divides it into 100 equal-sized tiles (10x10 grid) (following the 10x10 tiling preprocessing step),
    converts each tile to a 3-channel (RGB) image suitable for YOLO input, and saves each tile as a separate image file.

    Args:
        input_path (str): Path to the input grayscale image files

    Side Effects:
        Saves 100 image tiles as 'tile_0.jpg', 'tile_1.jpg', ..., 'tile_99.jpg' in the current directory.
    """
    # 1) Read image in grayscale
    img = cv2.imread(input_path, cv2.IMREAD_GRAYSCALE)

    # 2) Tile the image into 100 pieces
    h, w = img.shape
    tile_height = h // 10
    tile_width = w // 10

    # Crop to ensure exact 100 tiles
    cropped = img[:tile_height*10, :tile_width*10]

    # Reshape into 100 tiles
    tiles_gray = cropped.reshape(10, tile_height, 10, tile_width)
    tiles_gray = tiles_gray.transpose(0, 2, 1, 3)
    tiles_gray = tiles_gray.reshape(100, tile_height, tile_width)

    # Convert each tile to 3-channel for YOLO
    tiles_3channel = np.stack([cv2.cvtColor(tile, cv2.COLOR_GRAY2RGB) for tile in tiles_gray])

    # Save tiles as images
    for i, tile in enumerate(tiles_3channel):
        cv2.imwrite(f'tile_{i}.jpg', tile)

# Rosette Detection

In [4]:
def run_detection(model_path, cropped_dir):
  """
    Runs object detection on the 100 tile images generated through preprocessing, saves detected object crops, and cleans up temporary files.

    Args:
        model_path (str): Path to YOLO detection model weights file (.pt)
        cropped_dir (str): Directory to save detected object crops (organized in 'crops' subdirectory)

    Notes:
        Expects pre-generated tile images (tile_0.jpg to tile_99.jpg) in current directory
    """
  model = YOLO(model_path)

  save_dir = cropped_dir


  # Create the save directory if it doesn't exist
  os.makedirs(save_dir, exist_ok=True)

  # Run YOLO detection with custom save directory
  for i in range(100):
      results = model.predict(
          source=f'tile_{i}.jpg',
          save=True,
          save_crop=True,
          project=save_dir,
          name='crops',
          exist_ok=True
      )
      print(f"Saved crop {i} to {save_dir}")

  # Clean up tile images
  for i in range(100):
      os.remove(f'tile_{i}.jpg')

# Human-in-the-loop Verification

In [5]:
def verify_rosettes_colab(input_dir):
    """
    Provides an interactive Jupyter notebook interface for manually verifying/categorizing rosette images.

    Args:
        input_dir (str): Path to directory containing images to review

    Features:
    - Creates 'verified_rosettes' subdirectory for approved images
    - Displays images one at a time with Yes/No buttons
    - Copies approved images to output directory
    - Progressively cleans up reviewed images
    - Works with common image formats (PNG, JPG, BMP, TIFF, WEBP)
    """
    output_dir = os.path.join(input_dir, 'verified_rosettes')
    os.makedirs(output_dir, exist_ok=True)
    valid_exts = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', '.webp')
    image_files = [f for f in sorted(os.listdir(input_dir)) if f.lower().endswith(valid_exts)]
    idx = {'value': 0}  # mutable index for closure

    def show_next_image(_=None):
        clear_output(wait=True)
        if idx['value'] >= len(image_files):
            print("✅ All images reviewed.")
            return
        filename = image_files[idx['value']]
        img_path = os.path.join(input_dir, filename)
        display(widgets.Image(value=open(img_path, 'rb').read(), format='png', width=300))
        print(f"\n{'='*50}\nImage: {filename}")

        def on_yes(b):
            shutil.copy(img_path, output_dir)
            print(f"✅ Saved {filename}")
            idx['value'] += 1
            show_next_image()

        def on_no(b):
            print(f"❌ Skipped {filename}")
            idx['value'] += 1
            show_next_image()

        yes_btn = widgets.Button(description="Yes", button_style='success')
        no_btn = widgets.Button(description="No", button_style='danger')
        yes_btn.on_click(on_yes)
        no_btn.on_click(on_no)
        display(widgets.HBox([yes_btn, no_btn]))

    show_next_image()

# Segmentation on verified rosettes

In [6]:
def extract_bounding_boxes(model_path, input_directory, output_dir,
                           classes=None, imgsz=640, conf=0.2, iou=0.45, max_det=8):
    """
    Segments detected rosettes and extracts bounding boxes from YOLO segmentation model predictions and saves results as text files.

    Args:
        model_path (str): Path to the YOLO segmentation model weights
        input_directory (str): Directory containing images to process
        output_dir (str): Directory where the text files will be saved
        classes (list, optional): List of classes to detect. Default is all classes.
        imgsz (int): Image size for detection. Default is 640.
        conf (float): Confidence threshold. Default is 0.2.
        iou (float): IoU threshold. Default is 0.45.
        max_det (int): Maximum number of detections per image. Default is 8.

    Returns:
        dict: Summary of processed files containing:
              - total_processed: Number of processed images
              - total_detections: Total number of detections across all images
              - skipped_files: Number of files with no detections
    """
    import os
    from ultralytics import YOLO
    import numpy as np

    # Load segmentation model
    model = YOLO(model_path)

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Run prediction with segmentation model
    results = model.predict(
        input_directory,
        classes=classes,
        imgsz=imgsz,
        conf=conf,
        iou=iou,
        max_det=max_det,
        save=False
    )

    # Track statistics
    total_processed = 0
    total_detections = 0
    skipped_files = 0

    # Process results - get boxes directly
    for result in results:
        if not result.boxes:
            print(f"Skipping {result.path} - no objects detected")
            skipped_files += 1
            continue

        base_name = os.path.basename(result.path)
        txt_path = os.path.join(output_dir, f"{os.path.splitext(base_name)[0]}.txt")

        detection_count = 0
        with open(txt_path, 'w') as f:
            for box in result.boxes:
                # Get class and confidence
                cls = int(box.cls.item())
                conf = box.conf.item()

                # Get coordinates (xyxy format)
                x_min, y_min, x_max, y_max = box.xyxy[0].cpu().numpy().astype(int)

                # Write to file: class, confidence, x_min, y_min, x_max, y_max
                line = f"{cls} {conf:.4f} {x_min} {y_min} {x_max} {y_max}\n"
                f.write(line)
                detection_count += 1

        print(f"Saved: {txt_path} with {detection_count} detections")
        total_processed += 1
        total_detections += detection_count

    print(f"\nSummary:")
    print(f"- Processed {total_processed} images")
    print(f"- Found {total_detections} objects")
    print(f"- Skipped {skipped_files} images (no detections)")

    # Return summary
    return {
        "total_processed": total_processed,
        "total_detections": total_detections,
        "skipped_files": skipped_files
    }


# Use identified labels to generate data for each rosette

In [7]:
def process_detection_file(file_path):
    """
    Processes YOLO detection results file to extract structured metrics for rosette analysis.

    Args:
        file_path (str): Path to .txt detection file containing:
                         [class_id confidence xmin ymin xmax ymax] per line

    Returns:
        dict: Structured metrics containing:
            - Count of lobe detections
            - Best inner/outer boundary coordinates and confidence
            - Top 6 lobe coordinates and confidences (zero-padded if less than 6)

    File Format Expectations:
        Each line should contain 6 space-separated values:
        class_id (0=inner, 1=lobe, 2=outer) confidence xmin ymin xmax ymax
    """
    data = defaultdict(list)

    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 6:
                continue

            cls = int(parts[0])
            conf = float(parts[1])
            xmin = int(parts[2])
            ymin = int(parts[3])
            xmax = int(parts[4])
            ymax = int(parts[5])

            if cls == 0:
                data['inner'].append((conf, xmin, ymin, xmax, ymax))
            elif cls == 1:
                data['lobe'].append((conf, xmin, ymin, xmax, ymax))
            elif cls == 2:
                data['outer'].append((conf, xmin, ymin, xmax, ymax))

    # Process results
    result = {
        'lobe_count': len(data['lobe']),
        'inner_x_max': 0, 'inner_x_min': 0, 'inner_y_max': 0, 'inner_y_min': 0, 'inner_confidence': 0,
        'outer_x_max': 0, 'outer_x_min': 0, 'outer_y_max': 0, 'outer_y_min': 0, 'outer_confidence': 0
    }

    # Get best inner (highest confidence)
    if data['inner']:
        best_inner = sorted(data['inner'], key=lambda x: -x[0])[0]
        result.update({
            'inner_confidence': best_inner[0],
            'inner_x_max': best_inner[3],
            'inner_x_min': best_inner[1],
            'inner_y_max': best_inner[4],
            'inner_y_min': best_inner[2]
        })

    # Get best outer (highest confidence)
    if data['outer']:
        best_outer = sorted(data['outer'], key=lambda x: -x[0])[0]
        result.update({
            'outer_confidence': best_outer[0],
            'outer_x_max': best_outer[3],
            'outer_x_min': best_outer[1],
            'outer_y_max': best_outer[4],
            'outer_y_min': best_outer[2]
        })

    # Process lobes (sorted by confidence)
    sorted_lobes = sorted(data['lobe'], key=lambda x: -x[0])
    for i in range(6):
        lobe_key = f'lobe{i+1}'
        if i < len(sorted_lobes):
            lobe = sorted_lobes[i]
            result.update({
                f'{lobe_key}_x_max': lobe[3],
                f'{lobe_key}_x_min': lobe[1],
                f'{lobe_key}_y_max': lobe[4],
                f'{lobe_key}_y_min': lobe[2],
                f'{lobe_key}_confidence': lobe[0]
            })
        else:
            result.update({
                f'{lobe_key}_x_max': 0,
                f'{lobe_key}_x_min': 0,
                f'{lobe_key}_y_max': 0,
                f'{lobe_key}_y_min': 0,
                f'{lobe_key}_confidence': 0
            })

    return result

# Final Pipeline

In [17]:
from google.colab import drive
drive.mount('/content/drive')

#INCLUDE PATH TO ROOT OF DOWNLOADED GITHUB FOLDER
REPO_ROOT = Path('.../GithubFolder')

# Define your paths
input_image_path = REPO_ROOT / 'OriginalDataset/images/1985-Physco-60k-100nmMOD_jpg.rf.dc733288a112fe4307dca2bee5a373b7.jpg'
model_path = REPO_ROOT / 'Detection/Yolov9Detection/yolov9c_v9_25e_rosetteAugmented/weights/best.pt'
save_directory = '/content/cropped_images'

# 1. Preprocess the image (creates temporary tile images)
preprocess_image(input_image_path)

# 2. Run detection with model
run_detection(
    model_path=model_path,
    cropped_dir=save_directory
)

# 3. Human in the loop verification
verify_rosettes_colab(
    '/content/cropped_images/crops/crops/rosette_object'
)

✅ All images reviewed.


In [19]:
results = extract_bounding_boxes(
     model_path= REPO_ROOT / 'Segmentation/Yolov9_Segmentation/yolov9n_v3_seg_run/weights/best.pt',
     input_directory='/content/cropped_images/crops/crops/rosette_object/verified_rosettes',
     output_dir='/content/pixel_coord_labels',
     classes=[0, 1, 2]
)

input_dir = '/content/pixel_coord_labels'
output_csv = '/content/metrics/detection_metrics.csv'

# Create CSV header
header = ['text_file_name', 'lobe_count']
for cls_type in ['inner', 'outer']:
    header += [f'{cls_type}_x_max', f'{cls_type}_x_min',
              f'{cls_type}_y_max', f'{cls_type}_y_min',
              f'{cls_type}_confidence']

for lobe_num in range(1, 7):
    header += [f'lobe{lobe_num}_x_max', f'lobe{lobe_num}_x_min',
              f'lobe{lobe_num}_y_max', f'lobe{lobe_num}_y_min',
              f'lobe{lobe_num}_confidence']

# Run Analytics
all_results = []
for filename in os.listdir(input_dir):
    if filename.endswith('.txt'):
        file_path = os.path.join(input_dir, filename)
        file_data = process_detection_file(file_path)
        file_data['text_file_name'] = filename
        all_results.append(file_data)

# Ensure the output directory exists
output_dir = os.path.dirname(output_csv)
os.makedirs(output_dir, exist_ok=True)

# Write to Analytics CSV
with open(output_csv, 'w', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=header)
    writer.writeheader()
    for row in all_results:
        writer.writerow(row)

print(f"CSV generated at {output_csv}")


image 1/8 /content/cropped_images/crops/crops/rosette_object/verified_rosettes/tile_31.jpg: 608x640 1 inner, 6 lobes, 1 outer, 62.0ms
image 2/8 /content/cropped_images/crops/crops/rosette_object/verified_rosettes/tile_40.jpg: 640x640 7 lobes, 1 outer, 63.3ms
image 3/8 /content/cropped_images/crops/crops/rosette_object/verified_rosettes/tile_48.jpg: 640x640 7 lobes, 1 outer, 51.2ms
image 4/8 /content/cropped_images/crops/crops/rosette_object/verified_rosettes/tile_54.jpg: 640x640 1 inner, 6 lobes, 1 outer, 36.4ms
image 5/8 /content/cropped_images/crops/crops/rosette_object/verified_rosettes/tile_542.jpg: 640x608 7 lobes, 1 outer, 37.5ms
image 6/8 /content/cropped_images/crops/crops/rosette_object/verified_rosettes/tile_62.jpg: 640x640 1 inner, 6 lobes, 1 outer, 41.3ms
image 7/8 /content/cropped_images/crops/crops/rosette_object/verified_rosettes/tile_81.jpg: 640x640 7 lobes, 1 outer, 37.9ms
image 8/8 /content/cropped_images/crops/crops/rosette_object/verified_rosettes/tile_87.jpg: 640x